In [1]:
## Task 1: Dataset Understanding

import pandas as pd
from pathlib import Path

# Load the dataset
data_path = Path('..') / '..' / 'ai_project_synthetic_datasets' / 'part_3_nlp_sequence_modeling' / 'customer_support_text_classification.csv'
df = pd.read_csv(data_path)
# 

# Number of records
num_records = len(df)

# Target labels/classes
target_classes = df['sentiment_label'].unique().tolist()

# Sample text records
sample_records = df.head(3).to_dict('records')

# Average text length (assuming 'customer_message' is the text column, based on context)
if 'customer_message' in df.columns:
    df['text_length'] = df['customer_message'].apply(lambda x: len(str(x).split()))
    avg_text_length = df['text_length'].mean()
else:
    avg_text_length = "N/A"

# Class distribution
class_distribution = df['sentiment_label'].value_counts().to_dict()

print(f"Number of records: {num_records}")
print(f"Target classes: {target_classes}")
print(f"Sample records: {sample_records}")
print(f"Average text length: {avg_text_length}")
print(f"Class distribution: {class_distribution}")

Number of records: 1500
Target classes: ['neutral', 'positive', 'negative']
Sample records: [{'ticket_id': 'TKT00001', 'channel': 'chat', 'customer_message': 'I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.', 'sentiment_label': 'neutral', 'word_count': 18, 'urgent_flag': 1}, {'ticket_id': 'TKT00002', 'channel': 'phone', 'customer_message': 'I need information about the payment process.', 'sentiment_label': 'neutral', 'word_count': 7, 'urgent_flag': 0}, {'ticket_id': 'TKT00003', 'channel': 'email', 'customer_message': 'The refund process was fast and convenient. I appreciate the quick response.', 'sentiment_label': 'positive', 'word_count': 12, 'urgent_flag': 0}]
Average text length: 12.722666666666667
Class distribution: {'neutral': 524, 'negative': 497, 'positive': 479}


In [3]:
# Task 2 : Data Preprocessing

import re
import tensorflow as tf
#from tensorflow.keras.preprocessing.text import Tokenizer
#from tensorflow.keras.preprocessing.sequence import pad_sequences

# Dataset already loaded from previous cell, no need to reload
# (uses df defined in earlier cell)

# 1. Cleaning: Lowercasing and removing special characters
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Keep only letters and spaces
    return text

df['cleaned_text'] = df['customer_message'].apply(clean_text)

# 2. Tokenization using Keras (standard for sequence models)
max_vocab_size = 5000
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_text'])

# Convert to sequences of integers
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])

# 3. Padding/Truncating
max_length = 25  # Padding to 25 based on the average length we found (~12 words)
padded_sequences = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# Save intermediate dataset just in case
df.to_csv("customer_support_preprocessed.csv", index=False)

# Display a comparison for the first record
original_sample = df['customer_message'].iloc[0]
cleaned_sample = df['cleaned_text'].iloc[0]
sequence_sample = sequences[0]
padded_sample = padded_sequences[0].tolist()
word_index_len = len(tokenizer.word_index)

print(f"Original: {original_sample}")
print(f"Cleaned: {cleaned_sample}")
print(f"Tokenized Sequence: {sequence_sample}")
print(f"Padded Sequence: {padded_sample}")
print(f"Total Vocabulary Size: {word_index_len}")

Original: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Cleaned: i need information about the payment process my ticket number is  please respond as soon as possible
Tokenized Sequence: [5, 30, 137, 40, 2, 91, 34, 4, 7, 8, 3, 11, 13, 9, 14, 9, 15]
Padded Sequence: [5, 30, 137, 40, 2, 91, 34, 4, 7, 8, 3, 11, 13, 9, 14, 9, 15, 0, 0, 0, 0, 0, 0, 0, 0]
Total Vocabulary Size: 182


In [1]:
# Task 3: Text Vectorization

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Load preprocessed dataset
df = pd.read_csv("customer_support_preprocessed.csv")

# Ensure there are no nulls in the cleaned text (just in case)
df['cleaned_text'] = df['cleaned_text'].fillna('')

# 1. Bag of Words (BoW) Vectorization
bow_vectorizer = CountVectorizer(max_features=5000)
bow_matrix = bow_vectorizer.fit_transform(df['cleaned_text'])

# 2. TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_text'])

# Print information about the vectors
print(f"Bag of Words Matrix Shape: {bow_matrix.shape}")
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

# Show a small snippet of the vocabulary and TF-IDF scores for the first document
vocab = tfidf_vectorizer.get_feature_names_out()
first_doc_tfidf = tfidf_matrix[0].toarray()[0]

print("\nSample TF-IDF representation for Document 1 (Non-zero scores only):")
for i, score in enumerate(first_doc_tfidf):
    if score > 0:
        print(f"Word: '{vocab[i]:<15}' | TF-IDF Score: {score:.4f}")

Bag of Words Matrix Shape: (1500, 180)
TF-IDF Matrix Shape: (1500, 180)

Sample TF-IDF representation for Document 1 (Non-zero scores only):
Word: 'about          ' | TF-IDF Score: 0.2960
Word: 'as             ' | TF-IDF Score: 0.4553
Word: 'information    ' | TF-IDF Score: 0.3586
Word: 'is             ' | TF-IDF Score: 0.1288
Word: 'my             ' | TF-IDF Score: 0.1301
Word: 'need           ' | TF-IDF Score: 0.2875
Word: 'number         ' | TF-IDF Score: 0.1678
Word: 'payment        ' | TF-IDF Score: 0.3409
Word: 'please         ' | TF-IDF Score: 0.2017
Word: 'possible       ' | TF-IDF Score: 0.2276
Word: 'process        ' | TF-IDF Score: 0.2890
Word: 'respond        ' | TF-IDF Score: 0.2276
Word: 'soon           ' | TF-IDF Score: 0.2276
Word: 'the            ' | TF-IDF Score: 0.1004
Word: 'ticket         ' | TF-IDF Score: 0.1622


In [2]:
# Task 4: Baseline Model Training and Evaluation

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load preprocessed dataset
df = pd.read_csv("customer_support_preprocessed.csv")
df['cleaned_text'] = df['cleaned_text'].fillna('')

# Define features (X) and target (y)
X = df['cleaned_text']
y = df['sentiment_label']

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorize the text using TF-IDF
# Note: We fit ONLY on the training data to prevent data leakage
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Initialize and train the Logistic Regression baseline model
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = baseline_model.predict(X_test_tfidf)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Baseline Model Accuracy: {accuracy:.4f}")
print("-" * 55)
print("Classification Report:")
print(report)



Baseline Model Accuracy: 1.0000
-------------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



In [5]:
# Task 5: Sequence Modeling with LSTM

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
'''
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
'''

from sklearn.preprocessing import LabelEncoder

# Load data
#df = pd.read_csv("customer_support_preprocessed.csv")
df['cleaned_text'] = df['cleaned_text'].fillna('')

# Encode labels
encoder = LabelEncoder()
y = encoder.fit_transform(df['sentiment_label'])

# Tokenize and pad
max_vocab_size = 5000
max_length = 25
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_text'])
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])
X = tf.keras.preprocessing.sequence.pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build LSTM Model
vocab_size = len(tokenizer.word_index) + 1 # +1 for padding
embedding_dim = 32

model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Print model summary
model.summary()

# Train
print("\nTraining model...")
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)

# Evaluate
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"LSTM Model Test Accuracy: {accuracy:.4f}")

C:\Users\Adhik\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training model...
LSTM Model Test Accuracy: 1.0000
